# 14.10 YaRN如何兼顧近處與遠處？


同一把位置尺縮小，快組和慢組指針的差角都跟著變小。延長文章時，我們希望較遠的位置仍落在合適範圍，也希望近處的差異不要全被一起壓小。**YaRN**（Yet another RoPE extensioN method，另一種RoPE延伸方法）的做法是讓不同轉速的組承擔不同程度的縮放，再調整比對分數轉成注意力權重時的數值尺度。

先看第一個改動：依各組在原訓練長度裡能轉過多少圈來分組。沿用14.9的手算轉速與原來的八個位置：快組每個位置轉60度，八個位置共轉 `8×60=480` 度，約1.33圈，超過一圈；慢組只轉 `8×15=120` 度，是三分之一圈，連一圈也沒轉完。在原訓練範圍裡就轉過一圈以上的組，指針已經轉過每個方向；連一圈都沒轉完的組，延長後才會轉到原範圍以外的角度。所以快慢要相對原訓練範圍判斷，不能只看一個角度數字就替所有模型劃同樣界線。

YaRN的原則是：連一圈都沒轉完的組只做完整插值（像14.8那樣把位置除以2，轉角跟著減半），例如慢組，延長後的轉角就壓回原範圍；在原範圍轉了很多圈的組不插值，保留原轉速，因為模型靠這些組分辨鄰近位置的先後；圈數介於兩者之間的組兩者混合，圈數越少越靠近完整插值，越多越靠近原轉速。因而不是所有指針都除以同一個數。混合從幾圈開始、幾圈以上算「很多」，分別記成α、β；論文說兩者要按模型個別調整，作者對LLaMA系列模型實驗後選α=1、β=32。下面把α設成1、β設成2，再加一個每個位置轉90度、八個位置正好轉兩圈的組，算出三組的新轉角，並核對不到一圈的組延長後還在不在原範圍：


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
L, L_prime = 8, 16
s = L_prime / L
alpha, beta = 1.0, 2.0
for theta in (90.0, 60.0, 15.0):
    turns = L * theta / 360
    gamma = min(max((turns - alpha) / (beta - alpha), 0.0), 1.0)
    new_theta = (1 - gamma) * theta / s + gamma * theta
    print(theta, "度：原範圍轉", round(turns, 3), "圈", "γ =", round(gamma, 3), "新轉角", round(new_theta, 2), "度")
    if turns < 1:
        print("  延長後最遠轉", (L_prime - 1) * new_theta, "度", "原範圍不到", L * theta, "度")

程式沿用14.8的 `L`、`L_prime`（就是T、T′），`s` 是延長倍數2；`theta` 是這組每個位置轉幾度，`turns` 是它在原來八個位置裡轉幾圈，`alpha`、`beta` 就是α、β。β=2是為了讓八個位置的例子也有一組轉得夠多圈而自訂的。`gamma` 那行把 `(turns - alpha) / (beta - alpha)` 夾在0到1之間：`max(…, 0.0)` 把負數改成0，`min(…, 1.0)` 把超過1的改成1。`new_theta` 就是補充區的式子 `(1−γ)×θ/s+γ×θ`：γ為1保留原轉速，γ為0完整插值、除以s，中間按比例混合。輸出是：90度那組轉2圈、γ=1，保留90度；快組轉1.333圈、γ印成0.333（就是1/3），新轉角 `(2/3)×30+(1/3)×60=40` 度，介於完整插值的30度與原轉速60度之間；慢組只轉0.333圈、γ=0，變成 `15/2=7.5` 度。

`if turns < 1:` 那段只核對不到一圈的組。照14.8，原座標從0到不到8，所以慢組在原範圍的轉角是0到不到 `8×15=120` 度；完整插值後，最後一張卡（位置15）轉 `15×7.5=112.5` 度，仍在範圍內。若讓慢組的γ為0.5，每個位置轉 `0.5×7.5+0.5×15=11.25` 度，位置15就轉到 `15×11.25=168.75` 度，超出原範圍；這就是不到一圈的組要完整插值的原因。另兩組不必比：它們在原範圍已轉滿一圈，轉角超過360度只是繞回轉過的方向。和14.8一樣，這裡比的是範圍，不表示每個新角度都恰好出現過。

下圖上半部依各組在原訓練長度裡轉幾圈分成三列，α、β就是上面的兩條界線，箭頭通往各自的新轉速。下半部是另一個改動：先算出各候選位置的比對分數，再調整它們的尺度，最後交給softmax換成總和為一的注意力權重。上下兩部分一起使用，才是這裡介紹的YaRN。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-14-10-yarn-components.svg")))

為什麼還有下面那一步？延長文章後，我們仍希望模型利用讀到的內容，維持下一token的預測品質。分頻縮放處理了位置轉角，混合回來的內容卻還取決於各位置分到多少注意力權重。YaRN作者固定長文章與分頻規則，再比較不同注意力尺度的預測代價：適當尺度比不調整時低，於是按這個結果選尺度，讓內容的混合強度配合延長後的文字預測工作。

尺度如何改變混合？注意力會比較很多位置的分數，分數彼此差多少，會影響權重集中在少數位置還是分散到更多位置。例如兩個候選分數是1與0，softmax的權重約為0.73與0.27；若先都乘2，分數變成2與0，權重約為0.88與0.12。較高的一項仍是第一項，但它分到更大的比例。這個乘2只是展示尺度的作用，不是YaRN的預設值；下面一併算出論文的建議值：


In [ ]:
import math

import torch

for c in (1.0, 2.0, (0.1 * math.log(s) + 1) ** 2):
    weights = torch.tensor([1.0, 0.0]).mul(c).softmax(-1)
    print("分數乘", round(c, 4), "權重", weights.round(decimals=4))

前兩列就是上面的手算：乘1時約0.7311與0.2689，乘2時約0.8808與0.1192。第三個乘數來自YaRN論文替LLaMA擬合的建議：`√(1/t)=0.1×ln(s)+1`，分數再乘 `1/t`；`math.log` 是自然對數ln，`s` 沿用上一段的延長倍數2，所以 `1/t=(0.1×ln 2+1)²≈1.1434`，權重變成約0.7583與0.2417，只比不調整時稍微集中。這個值屬於論文的模型與配方，換了模型要重新選。

YaRN在改變位置規則之外，也調整這種注意力尺度。分頻縮放決定每組指針怎麼轉，尺度調整決定比對分數換成權重時有多集中，兩者處理不同部分。較集中不一定較正確：如果原本最高分的位置就不合適，放大分數仍可能偏向它。因此尺度要與模型的訓練及驗收配合，不能把它理解成把正確答案的權重直接調高。

實際採用這套設計時，要按模型以實驗選分組界線與注意力尺度，讓模型適應改過的計算，再檢查較長文章與原短任務。分頻處理保留了近處與遠處不同的旋轉特性，尺度調整則控制權重的分布；最後能使用多長的文章，仍由指定任務的回答決定。

練習把第二段程式的 `[1.0, 0.0]` 改成 `[0.0, 0.0]`，先預測三種乘數下的權重會不會偏向其中一個，再核對：三列都是各0.5。尺度能放大既有分數差，卻不會在相同分數之間憑空產生偏好。再把第一段程式的 `beta` 改成論文替LLaMA選的 `32.0`，先預測三組：慢組不受影響，仍是7.5度；快組的γ是 `(1.333−1)/(32−1)≈0.011`，新轉角約30.32度；90度那組的γ是 `1/31≈0.032`，約46.45度。八個位置裡沒有一組轉到32圈，所以三組都接近完整插值。

<details>
<summary>選讀：把兩個改動寫成算式與查來源</summary>

設延長倍數為s，某組原本每個位置的轉角為θ。完整位置插值的新轉角是θ/s，原轉速仍是θ。用介於0與1的係數γ連接兩者，可寫成`新轉角 = (1−γ) × θ/s + γ × θ`。γ為0表示完整插值，為1表示保留原轉速，中間值表示過渡。γ用來決定旋轉規則，並不是某個文字位置的注意力權重。論文的分組依據是這組在原訓練長度T裡轉了幾圈，`r=T/λ`，λ是轉一整圈需要的位置數（式10；論文把原訓練長度寫成L），也就是程式裡的 `turns`；r小於α時γ為0，大於β時γ為1，介於兩者之間時線性過渡（式11）。作者對LLaMA系列用α=1、β=32。

另一個改動可以寫成`新分數 = c × 原分數`，c是正的尺度係數，再將新分數交給softmax。論文把它寫成 `softmax(qᵀk/(t√|D|))`（式14），也就是 `c=1/t`，並建議 `√(1/t)=0.1 ln(s)+1`（式15）。相關的尺度直覺可回讀[14.3](https://birdhackor.github.io/tiny-perceptron-vlm/14.3.html)。以上是概念表示。

原始定義見[YaRN，arXiv:2309.00071v3，§3.2 Definition 1與§3.3 Definition 2](https://arxiv.org/pdf/2309.00071v3)。§3.3按不同延長倍數下較低的文字預測代價擬合尺度；附錄A.3固定LLaMA 7B、延長倍數8、896篇各16k token的RedPajama文章，比較不同尺度與不調整尺度的結果。這裡的代價依文章實際下一token的機率計算，目的和[5.1的下一字預測](https://birdhackor.github.io/tiny-perceptron-vlm/5.1.html)相同。訓練與不同任務的驗收見§4.1–4.4。作者的模型結果屬於其指定配方；更多查證在[來源筆記](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-revision-20261005/sources/context.md)。本節的分數、圖與程式是手工機制算例，沒有新訓練成果。

</details>
